# Clinicopathological and Molecular Characteristics of Second Primary Colorectal Cancer Exploration with `mlcroissant`
This notebook provides a guide for loading and exploring the FAIR² dataset using the `mlcroissant` library.

### Dataset Source
The dataset source is provided via a Croissant schema URL.

In [ ]:
# Ensure `mlcroissant` library is installed
!pip install mlcroissant

## 1. Data Loading
Load metadata and records from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd

# Define the dataset URL
croissant_url = 'https://sen.science/doi/10.71728/senscience.qs2f-h81p/fair2.json'

# Load the dataset metadata
dataset = mlc.Dataset(croissant_url)
metadata = dataset.metadata
print(f"{metadata.name}: {metadata.description}")

## 2. Data Overview
Review available record sets, fields, and their IDs.

We'll list all available record sets and fields by their `@id` as described in the Croissant schema.

In [ ]:
# List all available record sets and their fields by @id
record_sets = dataset.record_sets

print('Available record sets:')
for rs in record_sets:
    print(f"  RecordSet @id: {rs['@id']}, name: {rs.get('name', '<no name>')}")

# For each record set, print the available field (column) ids
for rs in record_sets:
    print(f"\nFields for RecordSet {rs['@id']}:")
    fields = rs.get('field', [])
    # 'field' might be a dict or a list of dicts
    if isinstance(fields, dict):
        fields = [fields]
    for f in fields:
        if isinstance(f, dict):
            print(f"  Field @id: {f.get('@id', '<no id>')}  --  name: {f.get('name', '<no name>')}")
        else:
            print(f"  Field reference: {f}")

## 3. Data Extraction
Load data from specific record set(s) into a DataFrame for analysis.

Use the record set and field `@id`s identified above. For demonstration, we will select the primary tabular record set (replace `@id` below if needed):

In [ ]:
# Identify main data record set (edit '@id' if your data uses a different one from the overview above).
# For FAIR² dataset, let's auto-select the first tabular record set.
all_record_sets = dataset.record_sets

# Heuristically choose the main record set (e.g. the one with most fields)
main_rs = None
max_fields = 0
for rs in all_record_sets:
    fields = rs.get('field', [])
    n_fields = len(fields) if isinstance(fields, list) else 1
    if n_fields > max_fields:
        main_rs = rs
        max_fields = n_fields

if not main_rs:
    raise ValueError('No record set found in dataset.')

main_record_set_id = main_rs['@id']
print(f"Main data record set @id: {main_record_set_id}")

# List of all record set @ids
record_set_ids = [rs['@id'] for rs in all_record_sets]

dataframes = {}
for record_set_id in record_set_ids:
    records = list(dataset.records(record_set=record_set_id))
    dataframes[record_set_id] = pd.DataFrame(records)
    print(f"Loaded {len(records)} records for record set {record_set_id}")

# Display columns of the main record set
print(f"\nAvailable columns (fields) in main record set '{main_record_set_id}':")
print(dataframes[main_record_set_id].columns.tolist())

dataframes[main_record_set_id].head()

## 4. Exploratory Data Analysis (EDA)
We now perform basic data processing steps such as filtering records, normalizing numeric fields, and grouping data by key attributes.

**All field variables are referenced by their `@id`s as per Croissant schema.**

In [ ]:
# Choose a numeric field for demonstration.
# We'll heuristically try to select a numeric column (e.g. 'Age', 'IntervalBetweenCancers'), searching by column name.
df = dataframes[main_record_set_id]

# Try to auto-detect a likely numeric field by checking dtypes or column names
possible_numeric_columns = [col for col in df.columns if any(x in col.lower() for x in ['age', 'interval', 'number', 'count', 'duration', 'years'])]

if possible_numeric_columns:
    numeric_field_id = possible_numeric_columns[0]  # You can override this if needed
else:
    numeric_field_id = df.select_dtypes('number').columns[0]

print(f"Using numeric field '@id': {numeric_field_id}")

# Remove records with missing or invalid numeric field
df_valid_numeric = df[pd.to_numeric(df[numeric_field_id], errors='coerce').notnull()]
df_valid_numeric[numeric_field_id] = pd.to_numeric(df_valid_numeric[numeric_field_id])

threshold = df_valid_numeric[numeric_field_id].quantile(0.2)  # 20th percentile for demo
filtered_df = df_valid_numeric[df_valid_numeric[numeric_field_id] > threshold]
print(f"Filtered records with {numeric_field_id} > {threshold:.2f}:")
print(filtered_df.head())

# Normalize the chosen numeric field
filtered_df[numeric_field_id + '_normalized'] = (
    (filtered_df[numeric_field_id] - filtered_df[numeric_field_id].mean()) /
    filtered_df[numeric_field_id].std()
)
print(f"\nNormalized '{numeric_field_id}' for filtered records:")
print(filtered_df[[numeric_field_id, numeric_field_id + '_normalized']].head())

# Select a grouping field: search for a common categorical field
possible_group_columns = [col for col in df.columns if any(x in col.lower() for x in ['sex', 'gender', 'location', 'msi', 'status', 'type', 'group', 'site', 'histology'])]
if possible_group_columns:
    group_field_id = possible_group_columns[0]
else:
    group_field_id = df.select_dtypes('object').columns[0]

print(f"\nGrouping field '@id': {group_field_id}")
if group_field_id in filtered_df.columns:
    grouped_df = filtered_df.groupby(group_field_id)[numeric_field_id].mean().to_frame()
    print(f"\nGrouped mean of '{numeric_field_id}' by '{group_field_id}':")
    print(grouped_df.head())

## 5. Visualization
Visualize data distributions or relationships between fields in the dataset.

For example, we create a histogram of our numeric variable and a boxplot grouped by the grouping field.

*Remember: All variables are referred to by their `@id` (column name as loaded above).*

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

# Histogram of filtered numeric field
plt.figure(figsize=(6, 4))
sns.histplot(filtered_df[numeric_field_id], bins=15, kde=True, color='skyblue')
plt.title(f"Distribution of {numeric_field_id}")
plt.xlabel(numeric_field_id)
plt.ylabel('Count')
plt.show()

# Boxplot of numeric field by group
plt.figure(figsize=(8, 4))
sns.boxplot(x=group_field_id, y=numeric_field_id, data=filtered_df)
plt.title(f"{numeric_field_id} by {group_field_id}")
plt.xticks(rotation=45)
plt.show()

## 6. Conclusion
In this notebook, we've demonstrated how to load and explore the FAIR² dataset using `mlcroissant`, referencing all dataset elements by their `@id`. We've shown how to extract tabular data, filter and normalize a numeric variable, group records by a key attribute, and visualize key distributions. This approach supports robust, reproducible data science and ensures consistency when working with Croissant-compliant datasets.